# Caso práctico TecnoForms: reconocimiento de dígitos manuscritos

**Objetivo:** clasificar dígitos escritos a mano (0-9) para automatizar el procesamiento de formularios.

**Modelos comparados:**
1. KNN (sklearn)
2. SVM (sklearn)
3. Red neuronal convolucional (CNN, TensorFlow/Keras)

**Dataset:** MNIST (70 000 imágenes de 28x28 en escala de grises).

## 1. Carga de datos y preprocesamiento

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score

mnist = fetch_openml('mnist_784', version=1, as_frame=False) #las imágenes ya vienen aplanadas (784)
x = mnist.data.astype('float32')
y = mnist.target.astype('int32')
print(x.shape)
print(y.shape)
print(x[0].shape)

In [ ]:
#PREPROCESAMIENTO
#1. Las imágenes ya están en escala de grises (1 canal)
#2. Normalizamos los píxeles de 0-255 a 0-1
x = x / 255

#3. Dividimos en entrenamiento (80%) y prueba (20%)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

#4. Para la CNN las imágenes NO van aplanadas: (N, 28, 28, 1)
x_train_img = x_train.reshape(-1, 28, 28, 1)
x_test_img = x_test.reshape(-1, 28, 28, 1)

print('x_train', x_train.shape, '| x_test', x_test.shape)
print('x_train_img', x_train_img.shape)

#Ejemplos del dataset
plt.figure(figsize=(10, 5))
for i in range(8):
  plt.subplot(2, 4, i+1)
  plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
  plt.title('Etiqueta: ' + str(y_train[i]))
  plt.axis('off')

## 2. Modelos de sklearn (KNN y SVM)

Cada modelo va en un **pipeline** (como en clase): `PCA` reduce las 784 variables a 50 y luego el clasificador.
Los hiperparámetros se ajustan con **validación cruzada** (`GridSearchCV`, 3 folds) sobre un subconjunto de 10 000 imágenes para que no tarde demasiado; luego el mejor modelo se entrena con todo el set de entrenamiento.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

#Subconjunto para ajustar hiperparámetros
rng = np.random.RandomState(42)
idx = rng.choice(len(x_train), 10000, replace=False)
x_tune, y_tune = x_train[idx], y_train[idx]

#KNN
knn_pipe = make_pipeline(PCA(n_components=50), KNeighborsClassifier())
knn_grid = GridSearchCV(knn_pipe, {'kneighborsclassifier__n_neighbors': [3, 5, 7]}, cv=3, scoring='accuracy')
knn_grid.fit(x_tune, y_tune)
print('KNN mejores parámetros:', knn_grid.best_params_)
print('KNN accuracy en validación cruzada:', knn_grid.best_score_)

#SVM
svm_pipe = make_pipeline(PCA(n_components=50), SVC())
svm_grid = GridSearchCV(svm_pipe, {'svc__C': [1, 5, 10], 'svc__kernel': ['rbf']}, cv=3, scoring='accuracy')
svm_grid.fit(x_tune, y_tune)
print('SVM mejores parámetros:', svm_grid.best_params_)
print('SVM accuracy en validación cruzada:', svm_grid.best_score_)

In [ ]:
#Entrenamos los mejores modelos con TODO el set de entrenamiento (la SVM puede tardar unos minutos)
knn_model = clone(knn_grid.best_estimator_)
knn_model.fit(x_train, y_train)

svm_model = clone(svm_grid.best_estimator_)
svm_model.fit(x_train, y_train)

y_pred_knn = knn_model.predict(x_test)
y_pred_svm = svm_model.predict(x_test)

print('Accuracy KNN:', accuracy_score(y_test, y_pred_knn))
print('Accuracy SVM:', accuracy_score(y_test, y_pred_svm))

## 3. Red neuronal convolucional (CNN)

Arquitectura: 3 bloques `Conv2D + MaxPooling2D` para extraer características, `Flatten` y capas densas para clasificar (10 salidas con `softmax`).
Se ajusta la **tasa de aprendizaje** del optimizador Adam usando un 10% de los datos de entrenamiento como validación.

In [ ]:
def crear_cnn(lr):
  model = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(28, 28, 1)),
      tf.keras.layers.Conv2D(16, 3, activation='relu'),
      tf.keras.layers.MaxPooling2D(), #reducimos la dimensionalidad del mapa de características
      tf.keras.layers.Conv2D(32, 3, activation='relu'),
      tf.keras.layers.MaxPooling2D(),
      tf.keras.layers.Conv2D(64, 3, activation='relu'),
      tf.keras.layers.MaxPooling2D(),
      tf.keras.layers.Flatten(), #aplanamos para procesar en capas densas
      tf.keras.layers.Dense(64, activation='relu'),
      tf.keras.layers.Dense(10, activation='softmax')
  ])
  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                loss='sparse_categorical_crossentropy', metrics=['accuracy'])
  return model

#Ajuste del hiperparámetro: tasa de aprendizaje
resultados_cnn = {}
for lr in [0.001, 0.0003]:
  print('\nEntrenando CNN con learning rate =', lr)
  m = crear_cnn(lr)
  h = m.fit(x_train_img, y_train, epochs=5, validation_split=0.1, verbose=1)
  resultados_cnn[lr] = (m, h)

mejor_lr = max(resultados_cnn, key=lambda k: resultados_cnn[k][1].history['val_accuracy'][-1])
print('\nMejor learning rate:', mejor_lr)
cnn_model, historial = resultados_cnn[mejor_lr]

In [ ]:
#Curvas de aprendizaje
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(historial.history['accuracy'], label='entrenamiento')
plt.plot(historial.history['val_accuracy'], label='validación')
plt.title('Accuracy por época')
plt.xlabel('época')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(historial.history['loss'], label='entrenamiento')
plt.plot(historial.history['val_loss'], label='validación')
plt.title('Loss por época')
plt.xlabel('época')
plt.legend()

y_pred_cnn = cnn_model.predict(x_test_img).argmax(1)
print('Accuracy CNN:', accuracy_score(y_test, y_pred_cnn))

## 4. Evaluación con el conjunto de test

In [ ]:
predicciones = {'KNN': y_pred_knn, 'SVM': y_pred_svm, 'CNN': y_pred_cnn}

for nombre, y_pred in predicciones.items():
  print('=' * 50)
  print(nombre)
  print(classification_report(y_test, y_pred))

In [ ]:
#Matrices de confusión
plt.figure(figsize=(18, 5))
for i, (nombre, y_pred) in enumerate(predicciones.items()):
  ax = plt.subplot(1, 3, i+1)
  ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues', colorbar=False)
  ax.set_title('Matriz de confusión - ' + nombre)
plt.tight_layout()

In [ ]:
#Tabla comparativa (métricas macro: promedio de las 10 clases)
tabla = pd.DataFrame({
    nombre: {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='macro'),
        'Recall': recall_score(y_test, y_pred, average='macro'),
        'F1-score': f1_score(y_test, y_pred, average='macro'),
        'Errores (de %d)' % len(y_test): int((y_test != y_pred).sum())
    } for nombre, y_pred in predicciones.items()
}).T
print(tabla.round(4))

mejor = tabla['Accuracy'].idxmax()
print('\nEl modelo que más acierta es:', mejor, 'con accuracy', round(tabla.loc[mejor, 'Accuracy'], 4))

tabla['Accuracy'].plot(kind='bar', ylim=(0.9, 1), title='Accuracy en test por modelo')
plt.ylabel('accuracy')
plt.xticks(rotation=0)

## 5. Resultados visuales (predicciones vs. etiquetas reales)

In [ ]:
y_pred_mejor = predicciones[mejor]

#Primeras 8 muestras del test (formato: real / predicho)
plt.figure(figsize=(10, 5))
for i in range(8):
  plt.subplot(2, 4, i+1)
  plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
  plt.title('Real: ' + str(y_test[i]) + ' | Pred: ' + str(y_pred_mejor[i]))
  plt.axis('off')
plt.suptitle('Predicciones del mejor modelo (' + mejor + ')')

#Ejemplos donde el mejor modelo se equivoca
errores = np.where(y_pred_mejor != y_test)[0][:8]
plt.figure(figsize=(10, 5))
for k, i in enumerate(errores):
  plt.subplot(2, 4, k+1)
  plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
  plt.title('Real: ' + str(y_test[i]) + ' | Pred: ' + str(y_pred_mejor[i]), color='red')
  plt.axis('off')
plt.suptitle('Errores del mejor modelo (' + mejor + ')')

## 6. Arquitectura de la red neuronal

In [ ]:
cnn_model.summary()

| Elemento | Detalle |
|---|---|
| Entrada | Imagen 28x28x1 (escala de grises, normalizada 0-1) |
| Capas convolucionales | Conv2D(16, 3x3) → Conv2D(32, 3x3) → Conv2D(64, 3x3), cada una seguida de MaxPooling2D |
| Capas densas | Flatten → Dense(64) → Dense(10) |
| Función de activación | ReLU en capas ocultas, Softmax en la salida |
| Función de pérdida | sparse_categorical_crossentropy |
| Optimizador | Adam (learning rate ajustado: 0.001 vs 0.0003) |
| Entrenamiento | 5 épocas, 10% de validación |

## 7. Complemento: K-Means (agrupamiento sin etiquetas)

K-Means no usa las etiquetas para entrenar: agrupa los dígitos por parecido. Las etiquetas reales solo se usan al final para medir qué tan bien coinciden los grupos con los dígitos.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

x_km = x_train[:10000]
y_km = y_train[:10000]

x_km_pca = PCA(n_components=50).fit_transform(x_km)
kmeans = KMeans(n_clusters=10, n_init=10, random_state=42)
grupos = kmeans.fit_predict(x_km_pca)

print('Adjusted Rand Index:', adjusted_rand_score(y_km, grupos))
print('Normalized Mutual Info:', normalized_mutual_info_score(y_km, grupos))

#Visualización en 2D (PCA a 2 componentes)
x_2d = PCA(n_components=2).fit_transform(x_km)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(x_2d[:, 0], x_2d[:, 1], c=grupos, cmap='tab10', s=3)
plt.title('Clustering de K-MEANS')
plt.subplot(1, 2, 2)
plt.scatter(x_2d[:, 0], x_2d[:, 1], c=y_km, cmap='tab10', s=3)
plt.title('DATOS REALES')

## 8. Procesamiento de un formulario (datos ficticios)

Simulamos una fila de un formulario con varios dígitos escritos a mano. Pasos: **escala de grises → normalización → segmentación de cada dígito → ajuste a 28x28 → clasificación con la CNN**.Si el recorte fuera mayor a 28 px, se redimensiona (por ejemplo con `cv2.resize`) antes de clasificar.

In [ ]:
def a_gris(img):
  #si la imagen viene a color (alto, ancho, 3) la pasamos a escala de grises
  if img.ndim == 3:
    return img[..., :3] @ np.array([0.299, 0.587, 0.114])
  return img

def normalizar(img):
  img = a_gris(img).astype('float32')
  return img / img.max() if img.max() > 0 else img

def segmentar_digitos(img, umbral=0.2, gap_min=3):
  #devuelve (inicio, fin) de columnas de cada dígito, separando por columnas vacías
  hay_tinta = (img > umbral).any(axis=0)
  tramos, inicio, fin, vacias = [], None, None, 0
  for j, t in enumerate(hay_tinta):
    if t:
      if inicio is None:
        inicio = j
      fin = j
      vacias = 0
    elif inicio is not None:
      vacias += 1
      if vacias >= gap_min:
        tramos.append((inicio, fin))
        inicio, vacias = None, 0
  if inicio is not None:
    tramos.append((inicio, fin))
  return tramos

def a_28x28(recorte, umbral=0.2):
  #quitamos filas vacías y centramos el dígito en un lienzo de 28x28
  filas = np.where(recorte.max(axis=1) > umbral)[0]
  recorte = recorte[filas.min():filas.max()+1]
  h, w = recorte.shape
  lienzo = np.zeros((28, 28), dtype='float32')
  y0, x0 = (28 - h) // 2, (28 - w) // 2
  lienzo[y0:y0+h, x0:x0+w] = recorte
  return lienzo

#Formulario ficticio: 6 dígitos del set de test en una sola fila
n = 6
formulario = np.hstack([x_test_img[i, :, :, 0] for i in range(n)])
formulario = normalizar(formulario)

tramos = segmentar_digitos(formulario)
digitos = np.array([a_28x28(formulario[:, a:b+1]) for a, b in tramos])
pred_form = cnn_model.predict(digitos.reshape(-1, 28, 28, 1)).argmax(1)

print('Dígitos encontrados:', len(tramos))
print('Reales:    ', y_test[:n])
print('Predichos: ', pred_form)

plt.figure(figsize=(10, 2))
plt.imshow(formulario, cmap='gray')
plt.title('Fila del formulario (ficticia)')
plt.axis('off')

plt.figure(figsize=(10, 2))
for i, d in enumerate(digitos):
  plt.subplot(1, len(digitos), i+1)
  plt.imshow(d, cmap='gray')
  plt.title('Pred: ' + str(pred_form[i]))
  plt.axis('off')

## 9. Conclusiones

Completar con los resultados obtenidos en la tabla comparativa (sección 4):
- Modelo con mayor accuracy y por cuánto superó a los otros.
- Dígitos que más se confunden según las matrices de confusión.
- Ventaja de la CNN: aprende las características de la imagen (bordes, curvas) por sí misma, mientras que KNN y SVM trabajan con píxeles reducidos por PCA.
- Tiempo de entrenamiento y predicción de cada modelo.